<a href="https://colab.research.google.com/github/Macharia-Maurice/MERN-chatapp/blob/main/Copy_of_loRA_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/My Drive/Datasets/archive.zip" -d "/content/dataset"


Mounted at /content/drive


In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
import os
from PIL import Image
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from transformers import TextStreamer
import random
import pandas as pd
from datasets import Dataset


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    "havoic206/testing_1",
    load_in_4bit=True  # or False for 16-bit LoRA
)
FastVisionModel.for_inference(model)


==((====))==  Unsloth 2025.3.18: Fast Qwen2 patching. Transformers: 4.49.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/204M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2VLForConditionalGeneration(
      (visual): Qwen2VisionTransformerPretrainedModel(
        (patch_embed): PatchEmbed(
          (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
        )
        (rotary_pos_emb): VisionRotaryEmbedding()
        (blocks): ModuleList(
          (0-18): 19 x Qwen2VLVisionBlock(
            (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
            (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
            (attn): VisionSdpaAttention(
              (qkv): lora.Linear(
                (base_layer): Linear(in_features=1280, out_features=3840, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1280, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(


In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

RuntimeError: Unsloth: You already added LoRA adapters to your model!

In [ ]:
# 3. Data Preparation

#Disease information dictionary
disease_info = {
    "Tomato Bacterial spot": {
        "symptoms": "Dark, raised spots on leaves, stems, and fruits. Spots may have yellow halos.",
        "remedies": "Remove infected plants, avoid overhead watering, use copper-based fungicides, practice crop rotation.",
        "prevention": "Use disease-free seeds, maintain good air circulation, sanitize garden tools."
    },
    "Tomato Septoria leaf spot": {
        "symptoms": "Small, dark spots with gray centers on the underside of leaves.",
        "remedies": "Remove infected leaves and debris, apply appropriate fungicides, and improve airflow.",
        "prevention": "Practice crop rotation, avoid overhead irrigation, and use resistant varieties if available."
    },
    "Tomato Early blight": {
        "symptoms": "Concentric rings (target-like patterns) on older leaves with brown lesions.",
        "remedies": "Remove affected foliage, apply fungicides such as chlorothalonil, and monitor weather conditions.",
        "prevention": "Rotate crops, maintain proper spacing, and use disease-resistant cultivars."
    },
    "Tomato Spider mites Two-spotted spider mite": {
        "symptoms": "Tiny, stippled spots on leaves, fine webbing on the underside, and eventual leaf bronzing.",
        "remedies": "Use insecticidal soap or neem oil, spray water to dislodge mites, and introduce natural predators.",
        "prevention": "Avoid excessive nitrogen fertilization, maintain humidity, and regularly inspect plants."
    },
    "Tomato healthy": {
        "description": "Uniform green color, no spots or lesions, normal leaf shape and robust plant structure.",
        "care": "Regular watering, balanced fertilization, adequate sunlight, and proper pruning for air circulation."
    },
    "Tomato Target Spot": {
        "symptoms": "Circular brown spots with concentric rings resembling a target.",
        "remedies": "Remove infected leaves, apply fungicides as directed, and ensure proper plant spacing.",
        "prevention": "Maintain dry foliage, avoid overhead watering, and use organic mulch to minimize soil splash."
    },
    "Tomato Late blight": {
        "symptoms": "Dark, water-soaked lesions that enlarge rapidly; white fungal growth underneath; brown lesions on stems/fruits.",
        "remedies": "Remove and destroy infected plants immediately, apply copper-based or chlorothalonil fungicides.",
        "prevention": "Increase plant spacing, avoid overhead irrigation, plant resistant varieties, and remove volunteer plants."
    },
    "Tomato Tomato mosaic virus": {
        "symptoms": "Mottled, irregularly discolored leaves, distorted growth, and reduced yield.",
        "remedies": "Remove and destroy infected plants; no direct cure is available.",
        "prevention": "Use virus-free seeds and transplants, control aphid populations, and practice crop rotation."
    },
    "Tomato Leaf Mold": {
        "symptoms": "Moldy, yellow spots on the underside of leaves with a grayish, fuzzy growth.",
        "remedies": "Remove affected leaves and improve air circulation, apply fungicides if needed.",
        "prevention": "Avoid overhead watering, ensure proper spacing, and use drip irrigation."
    },
    "Tomato Tomato Yellow Leaf Curl Virus": {
        "symptoms": "Yellowing and upward curling of leaves, stunted growth, and premature flower drop.",
        "remedies": "Remove infected plants to reduce spread; no cure exists once infected.",
        "prevention": "Use resistant varieties, control whitefly populations, and implement proper crop hygiene."
    }
}

# Map the directory names directly to the disease info keys
dir_to_disease_key = {
    "Tomato___Bacterial_spot": "Tomato Bacterial spot",
    "Tomato___Early_blight": "Tomato Early blight",
    "Tomato___Late_blight": "Tomato Late blight",
    "Tomato___Leaf_Mold": "Tomato Leaf Mold",
    "Tomato___Septoria_leaf_spot": "Tomato Septoria leaf spot",
    "Tomato___Spider_mites Two-spotted_spider_mite": "Tomato Spider mites Two-spotted spider mite",
    "Tomato___Target_Spot": "Tomato Target Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "Tomato Tomato Yellow Leaf Curl Virus",
    "Tomato___Tomato_mosaic_virus": "Tomato Tomato mosaic virus",
    "Tomato___healthy": "Tomato healthy"
}

# Default information for diseases not in the dictionary
default_info = {
    "symptoms": "Shows characteristic lesions or spots on leaves, stems, or fruits.",
    "remedies": "Remove infected plant parts, apply appropriate fungicides or treatments based on severity.",
    "prevention": "Maintain plant hygiene, avoid overhead watering, ensure proper spacing for air circulation."
}


# Default information for diseases not in the dictionary
default_info = {
    "symptoms": "Shows characteristic lesions or spots on leaves, stems, or fruits.",
    "remedies": "Remove infected plant parts, apply appropriate fungicides or treatments based on severity.",
    "prevention": "Maintain plant hygiene, avoid overhead watering, ensure proper spacing for air circulation."
}

# 4. Function to create dataset from directory structure with recommendations
def create_dataset_from_directory_with_recommendations(dataset_path):
    data = []

    # Get all class directories
    class_dirs = [
        d
        for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ]

    for class_dir in class_dirs:
        class_path = os.path.join(dataset_path, class_dir)

        # Format display name for readability (for display in output)
        display_name = class_dir.replace("___", " ").replace("_", " ").strip()

        # Get the proper disease info key using our mapping dictionary
        if class_dir in dir_to_disease_key:
            disease_key = dir_to_disease_key[class_dir]
            info = disease_info[disease_key]
        else:
            print(f"Warning: No mapping found for '{class_dir}', using default info")
            info = default_info

        # Format detailed response based on whether it's healthy or diseased
        if "healthy" in class_dir.lower():
            detailed_response = f"""This is a healthy tomato plant. {info.get('description', '')}

Care recommendations:
{info.get('care', 'Regular watering, balanced fertilization, and adequate sunlight.')}"""
        else:
            detailed_response = f"""This tomato plant shows signs of {display_name}.

Symptoms:
{info.get('symptoms', default_info['symptoms'])}

Remedies:
{info.get('remedies', default_info['remedies'])}

Prevention:
{info.get('prevention', default_info['prevention'])}"""

        # Get all image files in this class directory
        image_files = [
            f
            for f in os.listdir(class_path)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

        # Process each image
        for img_file in image_files:
            img_path = os.path.join(class_path, img_file)
            try:
                # Just verify the image is valid
                with Image.open(img_path) as img:
                    pass  # We just want to check if it opens
                # Store the path, class name, and detailed response
                data.append({
                    "image_path": img_path,
                    "class_name": display_name,
                    "detailed_response": detailed_response
                })
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                continue

    # Create a pandas DataFrame
    df = pd.DataFrame(data)

    # Convert to Hugging Face Dataset
    dataset = Dataset.from_pandas(df)
    return dataset


In [ ]:
train_dir = "/content/dataset/New Plant Diseases Dataset(Augmented)/train"
valid_dir = "/content/dataset/New Plant Diseases Dataset(Augmented)/valid"

train_dataset =create_dataset_from_directory_with_recommendations(train_dir)
eval_dataset = create_dataset_from_directory_with_recommendations(valid_dir)

In [ ]:
train_dataset[1]


In [ ]:
# 4. Format the dataset for vision-language instruction tuning
def convert_to_conversation(sample):
    # Load the image from path
    image = Image.open(sample["image_path"]).convert("RGB")

    # Randomly select one of several instructions for variety
    instructions = [
        "What disease, if any, is affecting this tomato plant leaf? Please provide detailed information about symptoms, remedies, and prevention.",
        "Identify the condition of this tomato leaf and provide care recommendations.",
        "Is this tomato leaf healthy or diseased? If diseased, what is the disease and how should it be treated?",
        "Diagnose any disease present in this tomato plant leaf and suggest treatment options.",
        "Analyze this tomato leaf image and tell me its condition along with detailed recommendations.",
    ]
    instruction = random.choice(instructions)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": image},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["detailed_response"]}]},
    ]
    return {"messages": conversation}

In [ ]:
# 5. Check what the model outputs before fine-tuning
FastVisionModel.for_inference(model)  # Enable for inference

# Get a random sample from the dataset
sample_idx = random.randint(0, len(train_dataset) - 1)
sample = train_dataset[sample_idx]
image = Image.open(sample["image_path"]).convert("RGB")
instruction = "What disease, if any, is affecting this tomato plant leaf?"

messages = [
    {
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": instruction}],
    }
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

# print(f"Testing model before fine-tuning on a '{sample['text']}' sample:")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1,
)
print("\n")

In [ ]:
# Create custom data collator that loads images on-the-fly
class TomatoDiseaseDataCollator(UnslothVisionDataCollator):
    def __call__(self, examples):
        # Convert paths to PIL images first
        converted_examples = []
        for example in examples:
            # Load images for each example
            sample = example.copy()
            image = Image.open(sample["image_path"]).convert("RGB")

            # Create prompt with random instruction
            instructions = [
                "What disease, if any, is affecting this tomato plant leaf? Please provide detailed information about symptoms, remedies, and prevention.",
                "Identify the condition of this tomato leaf and provide care recommendations.",
                "Is this tomato leaf healthy or diseased? If diseased, what is the disease and how should it be treated?",
                "Diagnose any disease present in this tomato plant leaf and suggest treatment options.",
                "Analyze this tomato leaf image and tell me its condition along with detailed recommendations.",
            ]
            instruction = random.choice(instructions)

            # Create conversation format
            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": instruction},
                        {"type": "image", "image": image},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": sample["detailed_response"]}],
                },
            ]
            converted_examples.append({"messages": conversation})

        # Use parent class to process the examples with images
        return super().__call__(converted_examples)

In [ ]:

# 6. Prepare training
FastVisionModel.for_training(model)  # Enable for training

# Split the dataset (90% train, 10% validation)
# dataset = dataset.shuffle(seed=42)  # Shuffle before splitting
# train_size = int(0.9 * len(dataset))
# train_dataset = dataset.select(range(train_size))
# eval_dataset = dataset.select(range(train_size, len(dataset)))

print(
    f"Training on {len(train_dataset)} samples, evaluating on {len(eval_dataset)} samples"
)



In [ ]:
FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator=TomatoDiseaseDataCollator(model, tokenizer),# Must use!
    train_dataset = train_dataset,
    # eval_dataset=eval_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        max_steps = 100,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",

        # eval_strategy="steps",
        # eval_steps=10,

        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "ttms",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 4,
        max_seq_length = 2048,
    ),
)

In [ ]:
# Train the model
trainer_stats = trainer.train()

In [ ]:
# Enable inference mode
FastVisionModel.for_inference(model)  # Enable for inference

print("\nTesting model on validation samples:")

# Select 5 random indices from the evaluation dataset
sample_indices = random.sample(range(len(eval_dataset)), 10)

# Iterate over the randomly selected samples
for i, idx in enumerate(sample_indices):
    eval_sample = eval_dataset[idx]
    image = Image.open(eval_sample["image_path"]).convert("RGB")
    ground_truth = eval_sample["class_name"]
    instruction = "What disease, if any, is affecting this tomato plant leaf? Provide the diagnosis."

    messages = [
        {
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": instruction}],
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    print(f"Sample {i+1}:")
    print(f"Ground truth: {ground_truth}")
    print("Model prediction:")

    text_streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=10,
        use_cache=True,
        temperature=1.5,
        min_p=0.1
    )
    print("\n")


In [ ]:
from google.colab import userdata
HF_TOKEN=userdata.get('HF_TOKEN')

In [ ]:
# 8. Save the model

# Save the LoRA adapters

# local
# model.save_pretrained("tomato_disease_lora_model")
# tokenizer.save_pretrained("tomato_disease_lora_model")

# push loRA to hugging face
model.push_to_hub("havoic206/TomatoLeafCare-Qwen2-vision-7b", token = HF_TOKEN) # Online saving
tokenizer.push_to_hub("havoic206/TomatoLeafCare-Qwen2-vision-7b", token = HF_TOKEN) # Online saving

# Optionally save a merged model in float16 (uncomment if needed)
# model.save_pretrained_merged("tomato_disease_model_merged_16bit", tokenizer)

# To upload to Hugging Face (uncomment and replace with your info)
# model.push_to_hub_merged("havoic206/tomato_disease_model", tokenizer, token="HF_TOKEN")

In [ ]:
# from PIL import Image

# image_path = "/content/1.png"

# # Load the image from disk
# image = Image.open(image_path).convert("RGB")

# # Define your instruction (adjust for your task)
# instruction = instruction = (
#     "You are a multimodal plant expert with extensive knowledge. "
#     "Please analyze the following tomato leaf image in detail. "
#     "Identify any disease present, suggest possible remedies, and provide general advice on plant care. "
#     "Include comparisons to similar diseases in other plants if applicable."
# )

# # Create the conversation messages as expected by your tokenizer
# messages = [
#     {"role": "user", "content": [
#         {"type": "image"},
#         {"type": "text", "text": instruction}
#     ]}
# ]

# # Generate the input text prompt using the chat template
# input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

# # Tokenize both the image and text prompt
# inputs = tokenizer(
#     image,
#     input_text,
#     add_special_tokens=False,
#     return_tensors="pt",
# ).to("cuda")

# text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# # Generate the output
# _ = model.generate(
#     **inputs,
#     streamer=text_streamer,
#     max_new_tokens=128,
#     use_cache=True,
#     temperature=1.1,
# )
